In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

Project root added: C:\Users\USER\Desktop\2026_japan
✓ Imports ready


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_MAPS_API_KEY")


In [3]:
df_cand_parking = pd.read_csv("../data/parking lots/topK_cand_parking_lots.csv")
df_cand_parking

,park_id,lat,lng,p_avail,drive_time,walk_time,detour_time,score,R,C_walk,C_detour,C_drive
0,way/1120633664,43.071634,141.353794,0.876722,1.278615,5.487554,4.735191,0.366462,0.998382,0.033304,0.024027,0.071413
1,way/508300771,43.073310,141.352910,0.877063,1.555025,4.005956,3.530002,0.366275,0.998966,0.019475,0.013821,0.088838
2,way/1360082065,43.071934,141.355062,0.876134,1.399108,4.751330,4.119459,0.366199,0.997377,0.026432,0.018813,0.079009


In [4]:
import datetime as dt
import requests
import polyline
import folium

def get_google_route(api_key, origin_lat, origin_lng, dest_lat, dest_lng,
                     mode="driving", departure_time=None, traffic_model="best_guess"):

    url = "https://maps.googleapis.com/maps/api/directions/json"

    params = {
        "origin": f"{origin_lat},{origin_lng}",
        "destination": f"{dest_lat},{dest_lng}",
        "mode": mode,              # driving / walking
        "key": api_key,
        "alternatives": "false",
    }

    # 只有 driving 才能帶 departure_time / traffic_model
    if mode == "driving" and departure_time is not None:
        if isinstance(departure_time, dt.datetime):
            params["departure_time"] = int(departure_time.timestamp())
        else:
            # 你也可以直接傳 int unix timestamp
            params["departure_time"] = int(departure_time)
        params["traffic_model"] = traffic_model

    r = requests.get(url, params=params, timeout=15)
    data = r.json()

    if data.get("status") != "OK":
        raise RuntimeError(f"Google API error: {data.get('status')} - {data.get('error_message')}")

    route = data["routes"][0]
    poly = route["overview_polyline"]["points"]
    coords = polyline.decode(poly)  # [(lat,lng), ...]

    leg = route["legs"][0]
    dist = leg["distance"]["value"]     # meters
    dur  = leg["duration"]["value"]     # seconds

    return coords, dist, dur

def draw_route(layer, coords, color, tooltip=None, dash_array=None, weight=5):

    folium.PolyLine(
        coords,
        color=color,
        weight=weight,
        opacity=0.85,
        dash_array=dash_array,   # 例如 "6,10" 畫虛線
        tooltip=tooltip
    ).add_to(layer)

In [ ]:
import datetime as dt
from src.geo.grid_to_latlng import *

anchors = [
        {"x": 24, "y": 151, "lat": 43.06918333153887, "lng": 141.35147072116592},  # 札幌站 
        {"x": 24, "y": 148, "lat": 43.07940372979633, "lng": 141.34225589803765},  # 北海道大學
        {"x": 26, "y": 153, "lat": 43.05798589528942, "lng": 141.35402112326315},  # 狸小路商店街
        {"x": 52, "y": 81, "lat": 43.1982317547878, "lng": 140.99403634015297},  # 小樽站
        {"x": 50, "y": 41, "lat": 43.188064114901195, "lng": 140.79455411455163},  # 余市站
        {"x": 182, "y": 186, "lat": 43.85360951281324, "lng": 141.52348814480132},  # 増毛町文化センター
]

mapper = GridLatLngMapper(anchors)

start_lat = 43.06918333153887
start_lng = 141.35147072116592
dest_lat, dest_lng = mapper.grid_to_latlng(26, 152)


# 你可用查詢時間 q.hhmm 組一個 datetime；這裡示範「現在」
departure_time = dt.datetime.now()

fmap = folium.Map(location=[dest_lat, dest_lng], zoom_start=14)

# 起終點 marker
folium.Marker([start_lat, start_lng], popup=f"Start").add_to(fmap)
folium.Marker([dest_lat,  dest_lng],  popup=f"Destination").add_to(fmap)

colors = ["red", "blue", "purple", "orange", "darkgreen"]

all_points = [(start_lat, start_lng), (dest_lat, dest_lng)]  # for fit_bounds

top1 = (df_cand_parking
        .sort_values(by=["score"],
                     ascending=[False])
        .iloc[0])  

pid  = (top1["park_id"])
plat = float(top1["lat"])
plng = float(top1["lng"])
color = colors[0]

fg = folium.FeatureGroup(name=f"park_id={pid} score={top1['score']:.3f}")

coords_drive, dist1, dur1 = get_google_route(
    api_key, start_lat, start_lng, plat, plng,
    mode="driving", departure_time=departure_time
)

coords_walk, dist2, dur2 = get_google_route(
    api_key, plat, plng, dest_lat, dest_lng,
    mode="walking"
)

draw_route(fg, coords_drive, color,
           tooltip=f"[DRIVE] park={pid} / {dur1/60:.1f}min / {dist1/1000:.2f}km",
           dash_array=None, weight=5)

draw_route(fg, coords_walk, color,
           tooltip=f"[WALK] park={pid} / {dur2/60:.1f}min / {dist2/1000:.2f}km",
           dash_array="6,10", weight=4)

folium.Marker(
    [plat, plng],
    popup=(f"park_id = {pid}<br>"
           f"score = {top1['score']:.3f}<br>"
           f"p_avail = {top1['p_avail']:.3f}<br>"
           f"drive ≈ {top1['drive_time']:.2f} / walk ≈ {top1['walk_time']:.2f}")
).add_to(fg)

fg.add_to(fmap)
all_points.extend([(plat, plng)])
all_points.extend(coords_drive)
all_points.extend(coords_walk)

# 自動縮放到全部路線都看得到
min_lat = min(p[0] for p in all_points); max_lat = max(p[0] for p in all_points)
min_lng = min(p[1] for p in all_points); max_lng = max(p[1] for p in all_points)
fmap.fit_bounds([[min_lat, min_lng], [max_lat, max_lng]])

folium.LayerControl(collapsed=False).add_to(fmap)
fmap.save("parking_topk_routes.html")
print("✓ 地圖已輸出: parking_topk_routes.html")

✓ 地圖已輸出: parking_topk_routes.html
